In [1]:
import tubesml as tml
import pandas as pd
import numpy as np

from source.report import _point_to_proba

from sklearn.metrics import brier_score_loss, mean_squared_error

from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler

In [2]:
df = pd.read_csv('data/processed/women_training.csv')

N_FOLDS = 5
kfolds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=13)

df_train, df_test = tml.make_test(df, test_size=0.2, random_state=34)

DROP = ["target", "target_points", "ID", "DayNum", "Team1", "Team2",
        'T1_Loc', 'T2_Loc',
                "T1_region", "T2_region", "Season", "delta_Loc",
                "Season", "competitive", "competitive_score",
                "delta_def_rating_diff", "delta_impact_diff",
                "T1_def_rating_diff", "T2_def_rating_diff"]

df_train.head()

df_train = df.copy()

## Feats cats

In [3]:
all_feats = [c for c in df_train if c not in DROP]
all_feats

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [4]:
all_delta = [c for c in df_train if c not in DROP and "delta" in c]

all_delta

['delta_Ast',
 'delta_Ast_TO_ratio',
 'delta_Ast_TO_ratio_diff',
 'delta_Ast_diff',
 'delta_Away',
 'delta_Blk',
 'delta_Blk_diff',
 'delta_DR',
 'delta_DR_diff',
 'delta_DR_opportunity',
 'delta_DR_opportunity_diff',
 'delta_Eff_FG_perc_diff',
 'delta_FG3_ratio',
 'delta_FG3_ratio_diff',
 'delta_FGA',
 'delta_FGA2',
 'delta_FGA2_diff',
 'delta_FGA3',
 'delta_FGA3_diff',
 'delta_FGA_diff',
 'delta_FGM',
 'delta_FGM2',
 'delta_FGM2_diff',
 'delta_FGM3',
 'delta_FGM3_diff',
 'delta_FGM_diff',
 'delta_FGM_no_ast',
 'delta_FGM_no_ast_diff',
 'delta_FTA',
 'delta_FTA_diff',
 'delta_FTM',
 'delta_FTM_diff',
 'delta_N_wins',
 'delta_OR',
 'delta_OR_diff',
 'delta_OR_opportunity',
 'delta_OR_opportunity_diff',
 'delta_OT_win',
 'delta_PF',
 'delta_PF_diff',
 'delta_Score',
 'delta_Score_diff',
 'delta_Stl',
 'delta_Stl_diff',
 'delta_TO',
 'delta_TO_diff',
 'delta_TO_perposs',
 'delta_TO_perposs_diff',
 'delta_Tot_Reb',
 'delta_Tot_Reb_diff',
 'delta_True_shooting_perc_diff',
 'delta_def_ratin

In [5]:
no_delta = [c for c in df_train if c not in DROP and "delta" not in c]
no_delta

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [6]:
seeds = [c for c in df_train if c not in DROP and "Seed" in c] + [c for c in df_train if "quality" in c] + [c for c in df_train if "stage" in c] + [c for c in df_train if "elo" in c]
seeds

['T1_Seed',
 'T2_Seed',
 'delta_Seed',
 'T1_quality',
 'T2_quality',
 'delta_quality',
 'stage_Round1',
 'stage_Round2',
 'stage_Round3',
 'stage_Round4',
 'stage_final',
 'stage_finalfour',
 'stage_impossible',
 'T1_elo',
 'T2_elo',
 'delta_elo']

In [7]:
no_seeds = [c for c in df_train if c not in DROP and "Seed" not in c]
no_seeds

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [8]:
feats_dict = {"all_feats": all_feats,
              "all_delta": all_delta, "no_delta": no_delta, "no_seeds": no_seeds, "seeds": seeds}

# Points predictions

## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMRegressor(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                              learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="l2")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "l2"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

[I 2026-03-07 16:56:54,590] A new study created in memory with name: no-name-91533aa7-0b86-47df-bde0-6c85bff439e2
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
in

Number of finished trials: 2000
Best trial: {'max_depth': 144, 'num_leaves': 53, 'reg_lambda': 8.885726284153476, 'reg_alpha': 2.600387041450046, 'colsample_bytree': 0.9074559985648463, 'subsample': 0.8299784877820622, 'min_child_weight': 192.85514300845676, 'feats': 'all_delta', 'clip_val': 21, 'padd': 0.041387752386437195}


In [11]:
0.183152

0.183152

In [12]:
study.trials_dataframe().sort_values('value', ascending=True).head(20)

,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1928,1928,0.139372,2026-03-07 17:32:44.332359,2026-03-07 17:33:02.081328,0 days 00:00:17.748969,21,0.907456,all_delta,144,192.855143,53,0.041388,2.600387,8.885726,0.829978,COMPLETE
1022,1022,0.139501,2026-03-07 17:16:21.949221,2026-03-07 17:16:36.034185,0 days 00:00:14.084964,23,0.919262,all_delta,127,188.754273,45,0.047955,5.817860,0.034770,0.416891,COMPLETE
1033,1033,0.139505,2026-03-07 17:16:31.696308,2026-03-07 17:16:47.126982,0 days 00:00:15.430674,24,0.900083,all_delta,135,189.156186,41,0.044805,5.683902,5.519723,0.411037,COMPLETE
1131,1131,0.139554,2026-03-07 17:18:12.895593,2026-03-07 17:18:29.801655,0 days 00:00:16.906062,22,0.854340,all_delta,132,191.938940,56,0.048039,6.246979,1.280587,0.433228,COMPLETE
1145,1145,0.139592,2026-03-07 17:18:28.483192,2026-03-07 17:18:42.011317,0 days 00:00:13.528125,22,0.852450,all_delta,133,192.323410,49,0.043627,2.297060,0.121085,0.432054,COMPLETE
1357,1357,0.139592,2026-03-07 17:22:24.011589,2026-03-07 17:22:39.720609,0 days 00:00:15.709020,21,0.872523,all_delta,109,192.325073,44,0.044191,8.853498,5.577900,0.776342,COMPLETE
1676,1676,0.139599,2026-03-07 17:28:06.224836,2026-03-07 17:28:23.528759,0 days 00:00:17.303923,21,0.872729,all_delta,117,182.164271,51,0.042196,10.478233,9.642000,0.446000,COMPLETE
931,931,0.139606,2026-03-07 17:14:42.957871,2026-03-07 17:14:57.571851,0 days 00:00:14.613980,21,0.900434,all_delta,140,192.108249,50,0.042876,13.523450,4.848467,0.413298,COMPLETE
1576,1576,0.139625,2026-03-07 17:26:21.094384,2026-03-07 17:26:38.442985,0 days 00:00:17.348601,23,0.888195,all_delta,141,194.612995,47,0.045498,2.303123,0.027972,0.421267,COMPLETE
1130,1130,0.139664,2026-03-07 17:18:12.455996,2026-03-07 17:18:26.412927,0 days 00:00:13.956931,22,0.856115,all_delta,133,191.559968,56,0.048364,6.708223,1.937490,0.400572,COMPLETE


## XGBoost

In [13]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBRegressor(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric=mean_squared_error)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {'verbose': False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2000
Best trial: {'max_depth': 3, 'reg_lambda': 44.062971221405036, 'reg_alpha': 11.413674009182976, 'colsample_bytree': 0.8544560837593326, 'colsample_bylevel': 0.5969234712473717, 'subsample': 0.7813041231083666, 'min_child_weight': 254.60687569207073, 'feats': 'all_delta', 'clip_val': 20, 'padd': 0.030643412875891236}


,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
1870,1870,0.138646,2026-03-07 21:26:54.215934,2026-03-07 21:28:08.148008,0 days 00:01:13.932074,20,0.596923,0.854456,all_delta,3,254.606876,0.030643,11.413674,44.062971,0.781304,COMPLETE
1874,1874,0.138688,2026-03-07 21:27:13.223324,2026-03-07 21:28:18.752762,0 days 00:01:05.529438,20,0.570239,0.840950,all_delta,3,255.061588,0.029089,9.373684,45.719880,0.782092,COMPLETE
1884,1884,0.138750,2026-03-07 21:28:08.154635,2026-03-07 21:29:10.440525,0 days 00:01:02.285890,20,0.579168,0.842101,all_delta,3,244.526456,0.032497,8.353480,43.341902,0.781647,COMPLETE
1924,1924,0.139044,2026-03-07 21:32:21.632753,2026-03-07 21:33:15.013291,0 days 00:00:53.380538,20,0.604211,0.845593,all_delta,6,238.626603,0.031591,11.451028,41.412549,0.797028,COMPLETE
1454,1454,0.139156,2026-03-07 20:32:56.090112,2026-03-07 20:33:47.650049,0 days 00:00:51.559937,21,0.611043,0.844253,all_delta,3,259.045718,0.011973,83.629621,11.222347,0.816105,COMPLETE
1648,1648,0.139242,2026-03-07 20:50:11.541375,2026-03-07 20:51:11.498557,0 days 00:00:59.957182,21,0.792809,0.824365,all_delta,3,293.909373,0.033196,4.929587,32.566654,0.889037,COMPLETE
1749,1749,0.139263,2026-03-07 21:03:06.635483,2026-03-07 21:03:51.088764,0 days 00:00:44.453281,21,0.573534,0.834458,all_delta,3,240.364782,0.032967,5.194224,39.775215,0.769315,COMPLETE
1694,1694,0.139337,2026-03-07 20:54:15.168664,2026-03-07 20:55:12.183892,0 days 00:00:57.015228,20,0.590641,0.836086,all_delta,3,297.325631,0.013543,2.067928,34.723636,0.769091,COMPLETE
421,421,0.139364,2026-03-07 18:35:54.857226,2026-03-07 18:36:55.370421,0 days 00:01:00.513195,20,0.506192,0.826772,all_delta,3,277.212424,0.014659,4.250272,58.647561,0.809873,COMPLETE
206,206,0.139390,2026-03-07 18:14:28.886849,2026-03-07 18:15:20.651479,0 days 00:00:51.764630,20,0.656715,0.767686,all_delta,3,189.527791,0.014369,33.110529,13.399360,0.818434,COMPLETE


## Ridge

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Ridge(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

[I 2026-03-07 23:00:34,483] A new study created in memory with name: no-name-31e46ab1-c04a-4899-a0b3-4ae655592049
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
in

Number of finished trials: 2000
Best trial: {'alpha': 21.832977504843903, 'feats': 'all_delta', 'clip_val': 29, 'padd': 0.0057516248250359565}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1318,1318,0.137389,2026-03-07 23:17:00.844203,2026-03-07 23:17:10.363204,0 days 00:00:09.519001,21.832978,29,all_delta,0.005752,COMPLETE
1443,1443,0.137389,2026-03-07 23:18:20.463860,2026-03-07 23:18:31.479294,0 days 00:00:11.015434,21.844903,29,all_delta,0.003638,COMPLETE
1750,1750,0.137389,2026-03-07 23:22:02.359264,2026-03-07 23:22:11.817792,0 days 00:00:09.458528,21.905997,29,all_delta,0.006487,COMPLETE
1502,1502,0.137389,2026-03-07 23:18:55.839459,2026-03-07 23:19:05.495744,0 days 00:00:09.656285,21.920002,29,all_delta,0.003799,COMPLETE
1511,1511,0.137389,2026-03-07 23:18:58.424234,2026-03-07 23:19:08.444121,0 days 00:00:10.019887,21.920907,29,all_delta,0.004553,COMPLETE
1747,1747,0.137389,2026-03-07 23:22:01.743040,2026-03-07 23:22:11.592279,0 days 00:00:09.849239,21.924115,29,all_delta,0.006513,COMPLETE
1929,1929,0.137389,2026-03-07 23:23:59.990756,2026-03-07 23:24:09.965863,0 days 00:00:09.975107,21.933659,29,all_delta,0.007756,COMPLETE
1930,1930,0.137389,2026-03-07 23:24:00.088002,2026-03-07 23:24:10.927615,0 days 00:00:10.839613,21.954954,29,all_delta,0.007475,COMPLETE
1925,1925,0.137389,2026-03-07 23:23:58.137980,2026-03-07 23:24:09.577443,0 days 00:00:11.439463,22.039774,29,all_delta,0.001577,COMPLETE
1873,1873,0.137389,2026-03-07 23:23:13.784114,2026-03-07 23:23:23.407891,0 days 00:00:09.623777,22.085128,29,all_delta,0.003890,COMPLETE


## Lasso

In [11]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Lasso(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [12]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `

Number of finished trials: 2000
Best trial: {'alpha': 0.10384021896343953, 'feats': 'all_delta', 'clip_val': 28, 'padd': 0.01584309458358785}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
900,900,0.137041,2026-03-07 23:46:45.517796,2026-03-07 23:46:57.176854,0 days 00:00:11.659058,0.103840,28,all_delta,0.015843,COMPLETE
1911,1911,0.137043,2026-03-07 23:58:39.957098,2026-03-07 23:58:50.546720,0 days 00:00:10.589622,0.102477,28,all_delta,0.015194,COMPLETE
1861,1861,0.137043,2026-03-07 23:57:50.752824,2026-03-07 23:58:01.550805,0 days 00:00:10.797981,0.107156,29,all_delta,0.013136,COMPLETE
1927,1927,0.137044,2026-03-07 23:58:47.557151,2026-03-07 23:58:58.409184,0 days 00:00:10.852033,0.113699,29,all_delta,0.015580,COMPLETE
1586,1586,0.137045,2026-03-07 23:54:41.549195,2026-03-07 23:54:52.273541,0 days 00:00:10.724346,0.112803,28,all_delta,0.018783,COMPLETE
1991,1991,0.137046,2026-03-07 23:59:34.101832,2026-03-07 23:59:43.787858,0 days 00:00:09.686026,0.110167,27,all_delta,0.015407,COMPLETE
1591,1591,0.137046,2026-03-07 23:54:44.098924,2026-03-07 23:54:55.730848,0 days 00:00:11.631924,0.104979,28,all_delta,0.017893,COMPLETE
902,902,0.137048,2026-03-07 23:46:46.209750,2026-03-07 23:46:57.546556,0 days 00:00:11.336806,0.122213,28,all_delta,0.015872,COMPLETE
1804,1804,0.137049,2026-03-07 23:57:17.918096,2026-03-07 23:57:29.926386,0 days 00:00:12.008290,0.108994,29,all_delta,0.016936,COMPLETE
901,901,0.137050,2026-03-07 23:46:45.594595,2026-03-07 23:46:54.965071,0 days 00:00:09.370476,0.101185,28,all_delta,0.017188,COMPLETE


# Probability Predictions


## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMClassifier(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                               learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "auc"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1, show_progress_bar=True)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

[I 2026-03-13 21:21:09,219] A new study created in memory with name: no-name-8e56e434-bbb8-47f7-b508-729ebc8be563


  0%|          | 0/1000 [00:00<?, ?it/s]

invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value encountered in cast
invalid value 

Number of finished trials: 1000
Best trial: {'max_depth': 95, 'num_leaves': 66, 'reg_lambda': 14.719924093662266, 'reg_alpha': 2.164545586887004, 'colsample_bytree': 0.8168061949366496, 'subsample': 0.7219421803912166, 'min_child_weight': 5.621234552221577, 'feats': 'all_delta'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,state
922,922,0.144462,2026-03-13 21:52:51.771948,2026-03-13 21:53:08.949536,0 days 00:00:17.177588,0.816806,all_delta,95,5.621235,66,2.164546,14.719924,0.721942,COMPLETE
267,267,0.144914,2026-03-13 21:28:01.866986,2026-03-13 21:28:55.758647,0 days 00:00:53.891661,0.809527,all_delta,78,10.190101,61,16.461098,87.167731,0.651962,COMPLETE
514,514,0.145045,2026-03-13 21:38:08.612455,2026-03-13 21:39:09.004014,0 days 00:01:00.391559,0.770335,all_delta,94,9.831144,58,16.709198,82.979269,0.610411,COMPLETE
753,753,0.145080,2026-03-13 21:47:30.594022,2026-03-13 21:48:07.680299,0 days 00:00:37.086277,0.796895,all_delta,93,9.797005,61,5.688887,70.814884,0.609597,COMPLETE
271,271,0.145114,2026-03-13 21:28:08.437712,2026-03-13 21:29:00.757955,0 days 00:00:52.320243,0.818543,all_delta,63,9.807685,61,16.426686,86.807268,0.594825,COMPLETE
230,230,0.145177,2026-03-13 21:26:52.796283,2026-03-13 21:27:15.599065,0 days 00:00:22.802782,0.793950,all_delta,70,12.238159,69,7.048448,88.206395,0.718330,COMPLETE
863,863,0.145195,2026-03-13 21:51:29.767009,2026-03-13 21:51:53.267631,0 days 00:00:23.500622,0.783656,all_delta,100,8.144396,32,13.585698,13.832254,0.740897,COMPLETE
804,804,0.145211,2026-03-13 21:49:10.029556,2026-03-13 21:50:07.408701,0 days 00:00:57.379145,0.792178,all_delta,109,6.386383,67,15.575415,72.125632,0.732063,COMPLETE
723,723,0.145218,2026-03-13 21:46:10.215761,2026-03-13 21:46:39.841009,0 days 00:00:29.625248,0.807658,all_delta,86,0.187516,60,5.207935,10.804254,0.623894,COMPLETE
801,801,0.145245,2026-03-13 21:49:03.584771,2026-03-13 21:49:35.750675,0 days 00:00:32.165904,0.792278,all_delta,109,5.267391,34,12.483079,18.200167,0.602833,COMPLETE


## XGBoost

In [11]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBClassifier(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            min_child_weight=param["min_child_weight"],
                            early_stopping_rounds=100,
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {"verbose": False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [12]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1, show_progress_bar=True)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

  0%|          | 0/1000 [00:00<?, ?it/s]

Number of finished trials: 1000
Best trial: {'max_depth': 45, 'reg_lambda': 24.630358567716815, 'reg_alpha': 0.005587822696346834, 'colsample_bytree': 0.6658977881487521, 'colsample_bylevel': 0.9475615687672185, 'subsample': 0.7770768291448855, 'min_child_weight': 9.433107795161856, 'feats': 'all_delta'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
903,903,0.143776,2026-03-13 22:32:46.000123,2026-03-13 22:33:12.690409,0 days 00:00:26.690286,0.947562,0.665898,all_delta,45,9.433108,0.005588,24.630359,0.777077,COMPLETE
559,559,0.143816,2026-03-13 22:19:03.551282,2026-03-13 22:19:38.469585,0 days 00:00:34.918303,0.800837,0.746775,all_delta,61,8.014450,1.987131,23.508076,0.922291,COMPLETE
393,393,0.144132,2026-03-13 22:10:51.303132,2026-03-13 22:11:26.923435,0 days 00:00:35.620303,0.826029,0.743185,all_delta,32,5.384464,1.605836,22.026646,0.880366,COMPLETE
260,260,0.144173,2026-03-13 22:05:49.317847,2026-03-13 22:06:26.012543,0 days 00:00:36.694696,0.684354,0.602876,all_delta,43,5.975424,4.622312,17.838183,0.940182,COMPLETE
526,526,0.144204,2026-03-13 22:17:24.100863,2026-03-13 22:18:03.613109,0 days 00:00:39.512246,0.795359,0.728007,all_delta,47,3.861392,0.211928,26.379952,0.902508,COMPLETE
628,628,0.144231,2026-03-13 22:21:36.492149,2026-03-13 22:22:10.494473,0 days 00:00:34.002324,0.869720,0.660611,all_delta,51,8.446858,3.598106,23.025779,0.880287,COMPLETE
391,391,0.144241,2026-03-13 22:10:47.992642,2026-03-13 22:11:23.972926,0 days 00:00:35.980284,0.825212,0.745827,all_delta,30,5.214469,1.737461,22.273242,0.885467,COMPLETE
557,557,0.144318,2026-03-13 22:18:56.284093,2026-03-13 22:19:27.545899,0 days 00:00:31.261806,0.799932,0.692452,all_delta,56,8.322699,2.191187,23.082972,0.925592,COMPLETE
386,386,0.144344,2026-03-13 22:10:38.306975,2026-03-13 22:11:13.895991,0 days 00:00:35.589016,0.830540,0.685485,all_delta,46,5.710445,1.767640,16.902740,0.870692,COMPLETE
635,635,0.144393,2026-03-13 22:21:48.145184,2026-03-13 22:22:19.274806,0 days 00:00:31.129622,0.601410,0.662717,all_delta,57,5.363013,3.618210,15.715432,0.882875,COMPLETE


## LogisticRegression

In [13]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        'C': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = LogisticRegression(C=param["C"], random_state=34, max_iter=10000, n_jobs=-1)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1, show_progress_bar=True)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

  0%|          | 0/2000 [00:00<?, ?it/s]

Number of finished trials: 2000
Best trial: {'reg_lambda': 0.0913483145895113, 'feats': 'all_delta'}


,number,value,datetime_start,datetime_complete,duration,params_feats,params_reg_lambda,state
484,484,0.138094,2026-03-13 22:44:52.014529,2026-03-13 22:45:01.149034,0 days 00:00:09.134505,all_delta,0.091348,COMPLETE
838,838,0.138094,2026-03-13 22:49:11.905421,2026-03-13 22:49:24.005335,0 days 00:00:12.099914,all_delta,0.095679,COMPLETE
1049,1049,0.138094,2026-03-13 22:52:04.904564,2026-03-13 22:52:14.527149,0 days 00:00:09.622585,all_delta,0.092837,COMPLETE
974,974,0.138094,2026-03-13 22:50:57.667963,2026-03-13 22:51:07.394553,0 days 00:00:09.726590,all_delta,0.092876,COMPLETE
1134,1134,0.138094,2026-03-13 22:52:56.091610,2026-03-13 22:53:09.983868,0 days 00:00:13.892258,all_delta,0.095385,COMPLETE
1042,1042,0.138094,2026-03-13 22:51:55.991872,2026-03-13 22:52:05.908748,0 days 00:00:09.916876,all_delta,0.089929,COMPLETE
904,904,0.138095,2026-03-13 22:50:06.695084,2026-03-13 22:50:18.268879,0 days 00:00:11.573795,all_delta,0.093979,COMPLETE
1613,1613,0.138095,2026-03-13 22:58:53.211167,2026-03-13 22:59:02.663289,0 days 00:00:09.452122,all_delta,0.093788,COMPLETE
201,201,0.138095,2026-03-13 22:41:22.980860,2026-03-13 22:41:36.546458,0 days 00:00:13.565598,all_delta,0.090294,COMPLETE
283,283,0.138095,2026-03-13 22:42:15.774101,2026-03-13 22:42:26.249937,0 days 00:00:10.475836,all_delta,0.096128,COMPLETE
